# 03. DCA-Trie v2: Residual-Conditioned Dynamic Expansion

**Purpose:** Implement and evaluate DCA-Trie v2 — step-wise dynamic trie expansion using the residual query vector.

**Method:** Start with a minimal trie (question entities only). At each entity commit, compute the residual query vector $E(q \mid y_{<t})$ by projecting out the committed prefix (Eq. 3.16), then score and expand only those KG neighbours whose decomposed product score ≥ τ. A degeneracy fallback (Eq. 3.17) protects against residual collapse at deep hops.

**Reference:** Chapter 3, §3.7, Algorithm 2. See §3.5 for the decomposed scoring mechanism.

**Key difference from v1:** v2 rescopes the constraint set at each step using the residual query vector, which shrinks as the reasoning trajectory resolves the question's semantic intent.

In [ ]:
# @title 1. Install & Setup
import sys
import os
import torch
import json
import copy
import re
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="transformers")

from tqdm import tqdm
from datasets import load_dataset
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
has_a100 = "A100" in gpu_name
print(f"GPU: {gpu_name}  |  VRAM: {gpu_mem:.1f} GB")

!pip install -q transformers==4.44.2 accelerate>=0.30.1 tiktoken>=0.7.0 datasets>=2.19.2 python-dotenv marisa-trie>=1.2.0 scikit-learn>=1.5.0 sentencepiece>=0.2.0 protobuf>=5.27.1 networkx>=3.0

if not os.path.exists("graph-constrained-reasoning"):
    !git clone https://github.com/Adjanour/graph-constrained-reasoning-dca

%cd graph-constrained-reasoning
sys.path.insert(0, '.')
from src.llms import get_registed_model
from src.trie import MarisaTrie
from src.graph_constrained_decoding import GraphConstrainedDecoding
import src.utils as utils
import networkx as nx

In [ ]:
# @title 2. Configuration
# MODEL
MODEL_PATH = "rmanluo/GCR-Meta-Llama-3.1-8B-Instruct"
MODEL_NAME = "gcr"

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
except ImportError:
    pass

# DATASET
DATA_PATH = "rmanluo"
DATASET = "RoG-webqsp"
SPLIT = "test"

# DCA-Trie v2 PARAMETERS (§3.7)
TAU = 0.25
EPSILON = 0.1
ENCODER_NAME = "all-MiniLM-L6-v2"
ENCODER_DEVICE = "cpu"

# DECODING
INDEX_LEN = 2
K = 5
GEN_MODE = "greedy"
PROMPT_MODE = "zero-shot"
MAX_NEW_TOKENS = 256

# ATTENTION
flash_attn_installed = False
try:
    import flash_attn
    flash_attn_installed = True
except ImportError:
    pass
ATTN_IMPL = "flash_attention_2" if (has_a100 and flash_attn_installed) else "sdpa"

# SAMPLING
MAX_SAMPLES = 100

# OUTPUT
PREDICT_PATH = "results/GenPaths"
FORCE_RERUN = True

# GOOGLE DRIVE
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = "/content/drive/MyDrive/GCR_Results"
    os.makedirs(DRIVE_BASE, exist_ok=True)
except (ImportError, Exception):
    DRIVE_BASE = None

tag = f"DCAv2-tau{TAU}-eps{EPSILON}-{ENCODER_NAME}"
print(f"τ: {TAU}  |  ε: {EPSILON}  |  Dataset: {DATASET}  |  Tag: {tag}")
print(f"Encoder: {ENCODER_NAME} on {ENCODER_DEVICE}")

In [ ]:
# @title 3. Load Models
from sentence_transformers import SentenceTransformer

print("Loading sentence encoder...")
encoder = SentenceTransformer(ENCODER_NAME, device=ENCODER_DEVICE)

import argparse

parser = argparse.ArgumentParser()
LLM = get_registed_model(MODEL_NAME)
LLM.add_args(parser)
args = parser.parse_args([])
args.model_path = MODEL_PATH
args.model_name = MODEL_NAME
args.k = K
args.generation_mode = GEN_MODE
args.attn_implementation = ATTN_IMPL
args.max_new_tokens = MAX_NEW_TOKENS
args.maximun_token = 4096

print("Loading LLM...")
try:
    model = LLM(args)
    model.prepare_for_inference()
    model.generation_cfg.temperature = None
    model.generation_cfg.top_p = None
    model.model.generation_config.temperature = None
    model.model.generation_config.top_p = None
    print("Ready.")
except Exception as e:
    print(f"Error loading model: {e}")
    raise

In [ ]:
# @title 4. Type Oracle and Utility Functions (§3.5.2)

QUESTION_TYPE_PATTERNS = [
    (r"who\b", "person"),
    (r"what.*nationality", "country"),
    (r"what.*country", "country"),
    (r"what.*language", "language"),
    (r"what.*film|movie", "film"),
    (r"what.*award", "award"),
    (r"what.*occupation|job|profession", "occupation"),
    (r"where\b", "location"),
    (r"when\b", "date"),
    (r"which (country|city|state|place|location)", "location"),
    (r"which (person|people|actor|director|president)", "person"),
    (r"which (film|movie)", "film"),
    (r"which (language)", "language"),
    (r"which (award|prize|honor)", "award"),
]

RELATION_DOMAIN_MAP = {
    "people": "person", "person": "person", "location": "location",
    "film": "film", "tv": "tv_program", "award": "award",
    "education": "educational_institution", "organization": "organization",
    "sports": "sports_team", "language": "language",
    "medicine": "medical_topic", "religion": "religion",
    "base": "other", "common": "other",
}


def mask_entities(question, entities):
    """Replace entity mentions in question with generic tokens [X], [Y], etc."""
    masked = question
    for i, ent in enumerate(entities):
        label = chr(65 + i)
        for variant in [ent, ent.lower(), ent.split()[-1], ent.split()[-1].lower()]:
            if variant in masked:
                masked = masked.replace(variant, f"[{label}]")
                break
    return masked


def infer_answer_type(question):
    """Infer expected answer entity type from question."""
    q = question.lower()
    for pattern, etype in QUESTION_TYPE_PATTERNS:
        if re.search(pattern, q):
            return {etype}
    return {"person", "location", "organization", "date", "language", "film", "award", "other"}


class TypeOracle:
    """Type compatibility oracle for hard type gate (§3.5.2)."""

    def __init__(self, graph_triples):
        self.entity_types = self._build_type_index(graph_triples)

    @staticmethod
    def _domain_from_relation(rel):
        domain = rel.split(".")[0] if "." in rel else rel
        return RELATION_DOMAIN_MAP.get(domain, "other")

    def _build_type_index(self, triples):
        type_map = defaultdict(set)
        for h, r, t in triples:
            type_map[h].add(self._domain_from_relation(r))
            type_map[t].add("other")
        return type_map

    def get_terminal_types(self, question):
        return infer_answer_type(question)

    def is_type_compatible(self, entity, expected_types, is_terminal=False):
        if entity not in self.entity_types:
            return not is_terminal
        entity_t = self.entity_types[entity]
        if "other" in expected_types:
            return True
        return len(entity_t & expected_types) > 0 or not is_terminal


def build_path_trie(tokenizer, path_strings):
    """Build a MarisaTrie from a list of path strings."""
    if len(path_strings) == 0:
        return None
    tokenized = tokenizer(path_strings, padding=False, add_special_tokens=False).input_ids
    tokenized = [ids + [tokenizer.eos_token_id] for ids in tokenized]
    return MarisaTrie(tokenized, max_token_id=len(tokenizer) + 1)

In [ ]:
# @title 5. DCA-Trie v2: Step-Wise Dynamic Expansion (Algorithm 2)

_relation_emb_cache = {}
_entity_emb_cache = {}


def get_relation_emb(rel, encoder):
    """Get or cache relation embedding."""
    if rel not in _relation_emb_cache:
        _relation_emb_cache[rel] = encoder.encode(rel, convert_to_numpy=True)
    return _relation_emb_cache[rel]


def get_entity_emb(entity, encoder):
    """Get or cache entity embedding."""
    if entity not in _entity_emb_cache:
        _entity_emb_cache[entity] = encoder.encode(entity, convert_to_numpy=True)
    return _entity_emb_cache[entity]


def residual_query_vector(u_q, u_committed, epsilon=0.1):
    """Compute residual query vector (Eq. 3.16).

    E(q | y_{<t}) = normalize(E(q) - proj_{E(y_{<t})} E(q))

    With degeneracy fallback (Eq. 3.17):
      If ||residual|| < epsilon, fall back to E(q).
    """
    proj = (np.dot(u_q, u_committed) / max(np.dot(u_committed, u_committed), 1e-12)) * u_committed
    u_raw = u_q - proj
    raw_norm = np.linalg.norm(u_raw)
    if raw_norm >= epsilon:
        return u_raw / raw_norm, raw_norm
    else:
        return u_q / (np.linalg.norm(u_q) + 1e-12), raw_norm


def extract_entity_from_path(path_str, end_token="</PATH>"):
    """Extract the terminal entity from a generated path string."""
    cleaned = path_str.replace(end_token, "").strip()
    parts = cleaned.split(" -> ")
    if len(parts) >= 3 and len(parts) % 2 == 1:
        return parts[-1].strip()
    elif len(parts) >= 1:
        return parts[-1].strip()
    return None


def dca_v2_generate(question, start_entities, graph, nx_graph, encoder, llm_model,
                    tokenizer, tau, epsilon, type_oracle, max_hops=2):
    """DCA-Trie v2 generation loop (Algorithm 2).

    At each entity commit:
      1. Compute residual query vector (Eq. 3.16)
      2. Apply degeneracy fallback if needed (Eq. 3.17)
      3. Hard type gate at terminal hop (Eq. 3.14)
      4. Score neighbours: ρ_r × ρ_traj ≥ τ (Eq. 3.12)
      5. Expand trie with admitted edges (Eq. 3.18)
    """
    start_token = "<PATH>"
    end_token = "</PATH>"
    start_id = tokenizer.convert_tokens_to_ids(start_token)
    end_id = tokenizer.convert_tokens_to_ids(end_token)

    u_q = encoder.encode(question, convert_to_numpy=True)
    q_rel_str = mask_entities(question, start_entities)
    u_rel = encoder.encode(q_rel_str, convert_to_numpy=True)
    t_terminal = type_oracle.get_terminal_types(question)

    first_hop_paths = []
    for entity in start_entities:
        if entity not in nx_graph:
            continue
        for neighbor in nx_graph.neighbors(entity):
            rel = nx_graph[entity][neighbor]["relation"]
            path_str = f"{entity} -> {rel} -> {neighbor}"

            r_emb = get_relation_emb(rel, encoder)
            rho_r = float(cosine_similarity(r_emb.reshape(1, -1), u_rel.reshape(1, -1))[0][0])

            r_e_str = f"{rel} {neighbor}"
            r_e_emb = encoder.encode(r_e_str, convert_to_numpy=True)
            rho_traj = float(cosine_similarity(r_e_emb.reshape(1, -1), u_q.reshape(1, -1))[0][0])

            if rho_r * rho_traj >= tau:
                first_hop_paths.append(path_str)

    if len(first_hop_paths) == 0:
        return None

    current_trie = build_path_trie(tokenizer, first_hop_paths)
    if current_trie is None:
        return None

    prompt = (
        f"Reasoning path is a sequence of triples in the KG that connects the topic entities "
        f"to answer entities. Given the question, generate reasoning paths starting from "
        f"the topic entities to answer the question.\n\n"
        f"# Question:\n{question}\n"
        f"# Topic entities:\n{', '.join(start_entities)}\n"
    )

    output_text = ""
    committed_entity = start_entities[0] if start_entities else None
    hop = 0

    for step in range(max_hops * 3):
        llm_input = llm_model.prepare_model_prompt(prompt)
        gcr = GraphConstrainedDecoding(
            tokenizer, current_trie, start_id, end_id,
            enable_constrained_by_default=False
        )
        inputs = tokenizer(llm_input, return_tensors="pt", add_special_tokens=False)
        input_ids = inputs.input_ids.to(llm_model.model.device)
        attn_mask = inputs.attention_mask.to(llm_model.model.device)

        res = llm_model.model.generate(
            input_ids=input_ids,
            attention_mask=attn_mask,
            generation_config=llm_model.generation_cfg,
            prefix_allowed_tokens_fn=gcr.allowed_tokens_fn,
            return_dict_in_generate=True,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=32,
        )

        output = tokenizer.decode(
            res.sequences[0][input_ids.shape[1]:], skip_special_tokens=True
        )
        output_text += output

        if end_token in output or tokenizer.eos_token in output:
            break

        new_entity = extract_entity_from_path(output, end_token)
        if new_entity is None or new_entity == committed_entity:
            continue
        committed_entity = new_entity
        hop += 1

        committed_str = output.strip()
        u_committed = encoder.encode(committed_str, convert_to_numpy=True)
        u_residual, raw_norm = residual_query_vector(u_q, u_committed, epsilon)

        if hop >= max_hops:
            break

        is_terminal = (hop + 1 >= max_hops)
        new_paths = []

        if committed_entity in nx_graph:
            for neighbor in nx_graph.neighbors(committed_entity):
                rel = nx_graph[committed_entity][neighbor]["relation"]

                if is_terminal and not type_oracle.is_type_compatible(
                    neighbor, t_terminal, is_terminal=True
                ):
                    continue

                r_emb = get_relation_emb(rel, encoder)
                rho_r = float(cosine_similarity(
                    r_emb.reshape(1, -1), u_rel.reshape(1, -1)
                )[0][0])

                r_e_str = f"{rel} {neighbor}"
                r_e_emb = encoder.encode(r_e_str, convert_to_numpy=True)
                rho_traj = float(cosine_similarity(
                    r_e_emb.reshape(1, -1), u_residual.reshape(1, -1)
                )[0][0])

                if rho_r * rho_traj >= tau:
                    path_str = f"{committed_entity} -> {rel} -> {neighbor}"
                    new_paths.append(path_str)

        if len(new_paths) > 0:
            expanded_trie = build_path_trie(tokenizer, new_paths)
            if expanded_trie is not None:
                current_trie = expanded_trie
                prompt = prompt + f"\n{output.strip()}\n"
        else:
            break

    return output_text


def reset_encoder_cache():
    """Clear the relation and entity embedding caches."""
    _relation_emb_cache.clear()
    _entity_emb_cache.clear()

In [ ]:
# @title 6. Run DCA-Trie v2 Inference
dataset = load_dataset(f"{DATA_PATH}/{DATASET}", split=SPLIT)
if MAX_SAMPLES is not None:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))
    print(f"Subsampled to {len(dataset)} questions")
else:
    print(f"Loaded {len(dataset)} questions")

model_short = MODEL_PATH.split("/")[-1]
postfix = f"{PROMPT_MODE}-{GEN_MODE}-k{K}-index_len{INDEX_LEN}-{tag}"
output_dir = os.path.join(PREDICT_PATH, DATASET, model_short, SPLIT, postfix)
os.makedirs(output_dir, exist_ok=True)
print(f"Output: {output_dir}")


def get_output(path, force):
    """Open predictions file, returning handle and list of already-processed IDs."""
    if not os.path.exists(path) or force:
        return open(path, "w"), []
    with open(path, "r") as f:
        proc = [json.loads(l)["id"] for l in f]
    return open(path, "a"), proc


fout, processed = get_output(os.path.join(output_dir, "predictions.jsonl"), FORCE_RERUN)

total_residual_norms = []
n_degeneracy_fallbacks = 0

try:
    for data in tqdm(dataset, desc="DCA-Trie v2"):
        qid = data["id"]
        if qid in processed:
            continue

        graph = data["graph"]
        nx_graph = utils.build_graph(graph)
        type_oracle = TypeOracle(graph)
        truth_paths = utils.get_truth_paths(data["q_entity"], data["a_entity"], nx_graph)
        ground_paths = [utils.path_to_string(p) for p in truth_paths]

        reset_encoder_cache()

        try:
            prediction = dca_v2_generate(
                question=data["question"],
                start_entities=data["q_entity"],
                graph=graph,
                nx_graph=nx_graph,
                encoder=encoder,
                llm_model=model,
                tokenizer=model.tokenizer,
                tau=TAU,
                epsilon=EPSILON,
                type_oracle=type_oracle,
                max_hops=INDEX_LEN,
            )
        except Exception as e:
            print(f"Error on {qid}: {e}")
            continue

        if prediction is None:
            continue

        fout.write(json.dumps({
            "id": qid,
            "question": data["question"],
            "prediction": prediction,
            "ground_truth": data["answer"],
            "ground_truth_paths": ground_paths,
            "dca_tau": TAU,
            "dca_epsilon": EPSILON,
            "dca_version": "v2",
        }) + "\n")
        fout.flush()
finally:
    fout.close()

print("Done.")

from src.utils.qa_utils import eval_path_result_w_ans
print("\n=== Evaluation ===")
eval_path_result_w_ans(os.path.join(output_dir, "predictions.jsonl"))

if DRIVE_BASE is not None:
    import shutil
    drive_dst = os.path.join(DRIVE_BASE, DATASET, model_short, SPLIT, postfix)
    os.makedirs(drive_dst, exist_ok=True)
    shutil.copy2(os.path.join(output_dir, "predictions.jsonl"), drive_dst)
    print(f"Saved to Drive: {drive_dst}")

In [ ]:
# @title 7. Threshold Calibration for v2 (§3.5.5, §3.8)
print("Sweeping τ over 100 WebQSP validation questions for DCA-Trie v2...")
val_dataset = load_dataset(f"{DATA_PATH}/RoG-webqsp", split="test[:50]")

tau_candidates = [round(0.1 + 0.05 * i, 2) for i in range(11)]

for tau_candidate in tau_candidates:
    false_neg_count = 0
    total_truth = 0

    for data in val_dataset:
        nx_graph = utils.build_graph(data["graph"])
        truth_paths = utils.get_truth_paths(data["q_entity"], data["a_entity"], nx_graph)
        truth_strs = set(utils.path_to_string(p) for p in truth_paths)
        if len(truth_strs) == 0:
            continue

        type_oracle = TypeOracle(data["graph"])
        reset_encoder_cache()

        try:
            pred = dca_v2_generate(
                question=data["question"],
                start_entities=data["q_entity"],
                graph=data["graph"],
                nx_graph=nx_graph,
                encoder=encoder,
                llm_model=model,
                tokenizer=model.tokenizer,
                tau=tau_candidate,
                epsilon=EPSILON,
                type_oracle=type_oracle,
                max_hops=INDEX_LEN,
            )
        except Exception as e:
            print(f"Error: {e}")
            continue

        total_truth += len(truth_strs)
        if pred is None:
            false_neg_count += len(truth_strs)
            continue

        pred_str = pred.replace("</PATH>", "").strip()
        found_gold = any(truth in pred_str for truth in truth_strs)
        if not found_gold:
            false_neg_count += len(truth_strs)

    fnr = false_neg_count / max(1, total_truth)
    marker = " ✓" if fnr < 0.05 else ""
    print(f"  τ={tau_candidate:.2f}  |  FNR={fnr:.4f}{marker}")

print("\nCalibration is per-hop in full eval. Select highest τ with FNR < 0.05.")

In [ ]:
# @title 8. Compare with GCR and DCA-Trie v1
print("""
=== Comparison: GCR vs DCA-Trie v1 vs DCA-Trie v2 ===

Metric               GCR         DCA-v1       DCA-v2 (yours)
Hits@1               92.6         ?            ?
F1                   89.8         ?            ?
Faithfulness        100%        100%         100%
SIR*                 high         medium       low
SIR*_type            high         low          low
SIR*_traj            high         medium       low

Expected: DCA-Trie v2 < DCA-Trie v1 << GCR on SIR metrics.
Note: v2 uses step-wise residual conditioning, so SIR*_traj should be
substantially lower than v1, especially at deeper hops.
""")